In [ ]:
# import the required libraries
import pandas as pd
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

# read the input data (cleaned embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title dataset is the dataset 
formed by merging & cleaning the embedded keywords and embedded title dataset.
"""
movies_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/old_final_mkg_dataset_bert.csv")

In [ ]:
movies_df.head(10)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,1114,"Documentary, Drama",tt0373861,mn,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,False,1115,Drama,tt0419773,de,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...
2,False,1116,"Drama, War",tt0460989,en,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...
3,False,1123,"Action, Drama, Thriller",tt0437232,en,Catch a Fire,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...
4,False,1124,"Drama, Mystery, Science Fiction",tt0482571,en,The Prestige,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...
5,False,1125,"Drama, Music",tt0443489,en,Dreamgirls,A trio of female soul singers cross over to th...,28.994,12/25/2006,English,6.750,947.0,1125.0,music record manager show business drug addict...,"[8, 18]",-1.34972304e-01 8.68575796e-02 2.66832620e-0...,7.76937753e-02 -8.23625207e-01 7.93465748e-0...
6,False,1126,"Drama, Mystery, Thriller",tt0385017,en,In My Father's Den,"Paul (Macfadyen), a prize-winning war journali...",8.440,10/7/2004,English,7.033,90.0,1126.0,sibling relationship soulmates funeral loss of...,"[8, 7, 10]",1.78888366e-01 3.32385972e-02 3.66470397e-0...,-7.42457658e-02 7.33588934e-02 -1.90228760e-0...
7,False,1127,Drama,tt0434292,es,Princesas,Caye is a young prostitute whose family is una...,7.356,9/2/2005,Spanish,6.750,76.0,1127.0,future depression friendship prostitute rape s...,[8],-3.02944463e-02 -7.09080845e-02 4.09498155e-0...,3.07188928e-02 1.43731758e-02 -5.90082929e-0...
8,False,1162,Drama,tt0056241,en,The Miracle Worker,"The true story of the frightening, lonely worl...",9.404,5/23/1962,English,7.800,280.0,1162.0,blindness and impaired vision deaf-mute educat...,[8],1.96001247e-01 4.75041538e-01 8.16400126e-0...,-1.14885390e-01 3.55399475e-02 -1.11388408e-0...
9,False,1163,"Drama, TV Movie",tt0246786,en,The Miracle Worker,"Devoted teacher Anne Sullivan leads deaf, blin...",10.345,11/12/2000,English,6.900,31.0,1163.0,blindness and impaired vision education woman ...,"[8, 14]",3.06614712e-02 4.18594122e-01 -2.77401090e-0...,-1.14885390e-01 3.55399475e-02 -1.11388408e-0...


In [ ]:
movies_df.shape[0]

234719

In [ ]:
movies_df.columns

Index(['adult', 'id', 'genres', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
drop_columns = ['adult', 'genres', 'original_language',
       'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro..."
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...


In [ ]:
# reduce the size of the overview column to 500 characters of string
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column '{}': {}".format('overview', max_length))
movies_df['overview'] = movies_df['overview'].str.slice(0, 500)
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column after strip '{}': {}".format('overview', max_length))

Maximum length of column 'overview': 1186.0
Maximum length of column after strip 'overview': 500.0


In [ ]:
# load the pre-trained bert tokenizer and model
model_name='bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

"""
:function: create_word_embeddings
:param sentence: takes a input sentence to be embedded i.e., movie overview
:return: embeded sentence vector of size 768
""" 
def create_word_embeddings(sentence):
    if pd.isna(sentence):
        return np.zeros(768)
    else:
      input_ids = tokenizer.encode(sentence, return_tensors='pt')
      with torch.no_grad():
          output = model(input_ids)
      embeddings = output.last_hidden_state.mean(dim=1).squeeze().numpy()
      return embeddings

# Apply the function to each row and create a new column 'keywords_embedding'
movies_df['overview_embedding'] = movies_df['overview'].apply(create_word_embeddings)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
movies_df['overview_embedding'] = movies_df['overview_embedding'].apply(lambda x: str(x))
movies_df.loc[0, 'overview_embedding']

'[ 3.44745591e-02 -2.66509261e-02 -2.78230965e-01 -5.36526702e-02\n  3.31405818e-01  3.90168071e-01  1.46535873e-01  3.00040394e-01\n  7.53507614e-02  8.83810520e-02 -1.86862782e-01 -2.25032076e-01\n  1.62307709e-01  5.38947821e-01 -5.10466039e-01  2.89864540e-01\n  3.88390243e-01  5.53409718e-02 -1.41214535e-01  2.47930184e-01\n -2.23860309e-01 -2.47946993e-01 -1.05556794e-01  5.74902117e-01\n  5.37896991e-01  1.67223692e-01  1.05287954e-01  2.45675147e-01\n -2.02234790e-01 -1.59001812e-01  1.57397583e-01  9.93357226e-02\n -2.07460612e-01 -2.20275894e-01 -2.00817451e-01  2.30230898e-01\n  7.95145333e-02 -2.33427539e-01  6.60995468e-02  3.24228793e-01\n -7.01512277e-01 -4.16964978e-01 -1.97174013e-01  1.47538468e-01\n -2.31593356e-01 -2.07164615e-01  2.48430386e-01  2.36640051e-01\n  3.05083364e-01 -1.42193124e-01 -4.46930259e-01  4.24459316e-02\n  4.61624451e-02 -2.37143964e-01  9.77546871e-02  4.09677297e-01\n -9.54824314e-02 -2.59163499e-01 -2.91868180e-01 -7.00283423e-02\n  7.70091

In [ ]:
# convert the embedded column array to embedded string and clean the overview_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_df['overview_embedding_string'] = movies_df['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '')

In [ ]:
movies_df.loc[0, 'overview_embedding_string']

' 3.44745591e-02 -2.66509261e-02 -2.78230965e-01 -5.36526702e-02  3.31405818e-01  3.90168071e-01  1.46535873e-01  3.00040394e-01  7.53507614e-02  8.83810520e-02 -1.86862782e-01 -2.25032076e-01  1.62307709e-01  5.38947821e-01 -5.10466039e-01  2.89864540e-01  3.88390243e-01  5.53409718e-02 -1.41214535e-01  2.47930184e-01 -2.23860309e-01 -2.47946993e-01 -1.05556794e-01  5.74902117e-01  5.37896991e-01  1.67223692e-01  1.05287954e-01  2.45675147e-01 -2.02234790e-01 -1.59001812e-01  1.57397583e-01  9.93357226e-02 -2.07460612e-01 -2.20275894e-01 -2.00817451e-01  2.30230898e-01  7.95145333e-02 -2.33427539e-01  6.60995468e-02  3.24228793e-01 -7.01512277e-01 -4.16964978e-01 -1.97174013e-01  1.47538468e-01 -2.31593356e-01 -2.07164615e-01  2.48430386e-01  2.36640051e-01  3.05083364e-01 -1.42193124e-01 -4.46930259e-01  4.24459316e-02  4.61624451e-02 -2.37143964e-01  9.77546871e-02  4.09677297e-01 -9.54824314e-02 -2.59163499e-01 -2.91868180e-01 -7.00283423e-02  7.70091861e-02  1.91241801e-01  1.1178

In [ ]:
drop_columns = ['overview_embedding']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview,overview_embedding_string
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,3.44745591e-02 -2.66509261e-02 -2.78230965e-0...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",-3.16557437e-01 -1.12028293e-01 2.61700094e-0...
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,-1.01136873e-02 1.58037364e-01 1.00762844e-0...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...,1.85870528e-01 1.91051021e-01 5.43662846e-0...


In [ ]:
movies_df['overview_embedding_string'] = movies_df['overview_embedding_string'].str.strip()

In [ ]:
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/old_final_mkg_dataset_bert.csv")

In [ ]:
merged_df = pd.merge(movies_df, movies_keywords_genres, on=['id', 'imdb_id', 'original_title'], how='inner')

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,3.44745591e-02 -2.66509261e-02 -2.78230965e-01...,False,"Documentary, Drama",mn,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",-3.16557437e-01 -1.12028293e-01 2.61700094e-0...,False,Drama,de,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,-1.01136873e-02 1.58037364e-01 1.00762844e-0...,False,"Drama, War",en,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...,False,"Action, Drama, Thriller",en,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...,1.85870528e-01 1.91051021e-01 5.43662846e-01...,False,"Drama, Mystery, Science Fiction",en,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...


In [ ]:
merged_df.shape[0]

234723

In [ ]:
merged_df.columns

Index(['id', 'imdb_id', 'original_title', 'overview_x',
       'overview_embedding_string', 'adult', 'genres', 'original_language',
       'overview_y', 'popularity', 'release_date', 'spoken_languages',
       'vote_average', 'vote_count', 'movie_id', 'keywords', 'genres_indices',
       'keywords_embedding_string', 'original_title_embedding_string'],
      dtype='object')

In [ ]:
nan_count = merged_df['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding_string' has {nan_count} NaN values.")
nan_count = merged_df['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
merged_df.dropna(subset=['overview_embedding_string'], inplace=True)
merged_df.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding_string' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
merged_df.shape[0]

234723

In [ ]:
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/final_mkg_dataset_bert.csv', index=False)

In [ ]:
#  Test

In [ ]:
movies_df.shape[0]

234719

In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,tt0373861,Die Geschichte vom weinenden Kamel,When a Mongolian nomadic family's newest camel...,3.44745591e-02 -2.66509261e-02 -2.78230965e-01...,False,"Documentary, Drama",mn,When a Mongolian nomadic family's newest camel...,5.016,6/29/2003,Mongolian,7.070,100.0,1114.0,shepherd tradition ritual violin pregnancy mot...,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,1115,tt0419773,Gespenster,"Nina, an end-of-teenage orphan with mental pro...",-3.16557437e-01 -1.12028293e-01 2.61700094e-0...,False,Drama,de,"Nina, an end-of-teenage orphan with mental pro...",4.417,9/15/2005,"German, French",6.300,26.0,1115.0,friendship berlin germany unsociability loss o...,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...
2,1116,tt0460989,The Wind That Shakes the Barley,In 1920s Ireland young doctor Damien O'Donovan...,-1.01136873e-02 1.58037364e-01 1.00762844e-0...,False,"Drama, War",en,In 1920s Ireland young doctor Damien O'Donovan...,16.052,6/23/2006,"Irish, Latin, English",7.400,747.0,1116.0,guerrilla warfare civil war sibling relationsh...,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...
3,1123,tt0437232,Catch a Fire,The true story of anti-apartheid activists in ...,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...,False,"Action, Drama, Thriller",en,The true story of anti-apartheid activists in ...,7.332,10/27/2006,"Afrikaans, English, Zulu",6.215,114.0,1123.0,police brutality right and justice resistance ...,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...
4,1124,tt0482571,The Prestige,A mysterious story of two magicians whose inte...,1.85870528e-01 1.91051021e-01 5.43662846e-01...,False,"Drama, Mystery, Science Fiction",en,A mysterious story of two magicians whose inte...,60.323,10/19/2006,English,8.203,15214.0,1124.0,dying and death suicide class society magic co...,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...


In [ ]:
# merge the embedded keywords, title and overview columns to form the input bert embedded dataset
drop_columns = ['imdb_id', 'original_title', 'overview_x', 'adult', 'original_language', 'overview_y', 'popularity', 'spoken_languages', 'vote_average', 'vote_count', 'movie_id', 'keywords']
merged_df = merged_df.drop(columns=drop_columns)
merged_df.head(5)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,3.44745591e-02 -2.66509261e-02 -2.78230965e-01...,"Documentary, Drama",6/29/2003,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,1115,-3.16557437e-01 -1.12028293e-01 2.61700094e-0...,Drama,9/15/2005,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...
2,1116,-1.01136873e-02 1.58037364e-01 1.00762844e-0...,"Drama, War",6/23/2006,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...
3,1123,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...,"Action, Drama, Thriller",10/27/2006,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...
4,1124,1.85870528e-01 1.91051021e-01 5.43662846e-01...,"Drama, Mystery, Science Fiction",10/19/2006,"[8, 7, 4]",3.90980601e-01 2.90082335e-01 5.26184797e-0...,3.64068747e-02 -1.96709305e-01 -4.71620530e-0...


In [ ]:
merged_df.shape[0]

234723

In [ ]:
# export the embedded keywords, title and overview dataset
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/f-final_mkg_dataset_bert.csv', index=False)

In [ ]:
# Testing read

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/bert-embed/input/f-final_mkg_dataset_bert.csv")

In [ ]:
test_df.shape[0]

234723

In [ ]:
test_df.columns

Index(['id', 'overview_embedding_string', 'genres', 'release_date',
       'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
test_df.head(4)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,1114,3.44745591e-02 -2.66509261e-02 -2.78230965e-01...,"Documentary, Drama",6/29/2003,"[11, 8]",2.24766999e-01 1.48856416e-01 -4.26082835e-0...,-3.27251524e-01 -1.14664100e-01 -1.55884609e-0...
1,1115,-3.16557437e-01 -1.12028293e-01 2.61700094e-0...,Drama,9/15/2005,[8],8.62499252e-02 6.64695725e-02 4.50119734e-0...,-3.33471775e-01 -3.40431213e-01 1.28486112e-0...
2,1116,-1.01136873e-02 1.58037364e-01 1.00762844e-0...,"Drama, War",6/23/2006,"[8, 16]",2.55626798e-01 3.36861275e-02 -1.86209410e-0...,-1.84306309e-01 7.14582950e-02 -1.47662491e-0...
3,1123,-1.07681625e-01 -8.75450373e-02 -9.33825895e-0...,"Action, Drama, Thriller",10/27/2006,"[19, 8, 10]",-1.35598034e-01 -3.07193458e-01 -3.09346825e-0...,4.00660694e-01 -1.56783655e-01 -1.28297895e-0...
